# auto-clean — S3 Loader

Browse your S3 bucket, pick a file, clean it, and optionally upload results back.

In [ ]:
# ── Configuration ────────────────────────────────
import sys
sys.path.append("..")

BUCKET = "your-bucket-name"   # change this
PREFIX = ""                    # optional folder prefix e.g. "raw/2025/"

In [ ]:
# ── List available files ────────────────────────
from src.s3 import list_bucket_files, print_bucket_files

files = list_bucket_files(BUCKET, prefix=PREFIX)
print_bucket_files(files, BUCKET)

In [ ]:
# ── Pick a file ─────────────────────────────────
# Set the number from the table above
SELECTED = 1

chosen = files[SELECTED - 1]
print(f"Selected: {chosen[chr(107)]}")

In [ ]:
# ── Download from S3 ────────────────────────────
from src.s3 import download_from_s3

local_path = download_from_s3(BUCKET, chosen["key"])
print(f"Local path: {local_path}")

In [ ]:
# ── Run auto-clean pipeline ─────────────────────
import pandas as pd
from pathlib import Path
from src.config import load_config
from src.profiler import profile_dataframe
from src.cleaner import clean
from src.reporter import generate_report
from src.loader import load_file, get_excel_profile
from rich.console import Console

console = Console()
config = load_config("../config.yaml")
ext = Path(local_path).suffix.lower()

# Excel: show structure first
if ext in {".xlsx", ".xls"}:
    excel_profile = get_excel_profile(local_path)
    from src.s3 import print_bucket_files
    for i, s in enumerate(excel_profile.sheets, 1):
        print(f"{i}. {s.name} — {s.total_rows} rows, header at row {s.detected_header_row}")

    SHEET_NUMBER = 1  # change if needed
    sheet = excel_profile.sheets[SHEET_NUMBER - 1]
    df = load_file(local_path, encoding=config.encoding,
                   sheet_name=sheet.name, header_row=sheet.detected_header_row)
else:
    df = load_file(local_path, encoding=config.encoding)

console.print(f"Loaded: {len(df)} rows x {len(df.columns)} columns")

In [ ]:
# ── Profile + Clean ─────────────────────────────
profile = profile_dataframe(df)
result = clean(df, config, profile)

print(f"Rows before : {result.rows_before}")
print(f"Rows after  : {result.rows_after}")
print(f"Duplicates  : {result.duplicates_removed}")
print(f"Nulls filled: {result.nulls_filled}")
print(f"Outliers    : {result.outliers_removed}")

In [ ]:
# ── Save outputs locally ────────────────────────
import os
os.makedirs("../output", exist_ok=True)

result.df_cleaned.to_csv("../output/cleaned.csv", index=False)
generate_report(profile, result, "../output")
print("Saved: ../output/cleaned.csv")
print("Saved: ../output/report.html")

In [ ]:
# ── Upload cleaned CSV back to S3 (optional) ────
# Set UPLOAD = True to enable
UPLOAD = False

if UPLOAD:
    from src.s3 import upload_to_s3
    original_key = chosen["key"]
    clean_key = original_key.replace("raw/", "clean/").replace(
        Path(original_key).suffix, "_cleaned.csv"
    )
    upload_to_s3("../output/cleaned.csv", BUCKET, clean_key)